# 3-Stage 평가서버 호환 베이스라인 — 학습

공개 예제 각 5건으로 모델을 학습하고 실제 참가자 제출구조와 동일한 경로에 체크포인트를 저장합니다.

In [ ]:
%pip install -r requirements.txt

## 1. 라이브러리·모델 구조·학습함수

In [ ]:
from pathlib import Path
import itertools, json, os, random, shutil
import cv2, numpy as np, pandas as pd, torch
from PIL import Image
from torch import nn
from torchvision import transforms as tv_transforms
from torchvision.models import resnet18
from torchvision.models.video import mvit_v2_s
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_320_fpn,
)


In [ ]:
ROOT=Path.cwd(); DATA=ROOT/'data'; MODEL=ROOT/'model'
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(20260825); random.seed(20260825)

# Stage1: 실제 제출 점수(0.316)가 팀 베이스라인 MViT(0.53397)보다 낮게 나와서 원복함 -
# 핸드크래프트 피처+로지스틱회귀는 합성 재녹화에만 과적합된 것으로 보임(실제 재녹화
# 데이터로 검증 전까지는 미사용). 아래 fit_stage1()은 MViT, 실험용 핸드크래프트 버전은
# fit_stage1_experimental_handcrafted()로 따로 둠(기본 실행 대상 아님).
SIZE=224
S1_MEAN=torch.tensor([0.45,0.45,0.45])[:,None,None,None]
S1_STD=torch.tensor([0.225,0.225,0.225])[:,None,None,None]
FEATURE_DIM=5  # 실험용 핸드크래프트 피처 차원

# Stage2: 충돌/진입 시점 (COCO 사전학습 탐지기 + 학습형 재정렬기)
VEHICLE_CLASSES={'car','motorcycle','bus','truck'}
SCORE_THR=0.2  # 0.5->0.2: 358영상/16248프레임 캐시 검증(mean IoU 0.378->0.442, 히트율 45.2%->53.1%, 0.2 밑은 수확체감)
RERANK_FEATURES=['cx','cy','bw','bh','score','aspect']

# Stage3: 가감속/조향 (optical flow 휴리스틱, 학습 없음 - 임계값만 보정)
ACCEL=['ACCELERATING','DECELERATING','CONSTANT','STOPPED']
STEER=['LEFT','STRAIGHT','RIGHT']
FLOW_SIZE=(160,90)


In [ ]:
def load_frames(path):
    cap=cv2.VideoCapture(str(path)); out=[]
    while True:
        ok,bgr=cap.read()
        if not ok: break
        out.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    cap.release()
    if not out: raise ValueError(f'cannot decode: {path}')
    return out

def sample_frames(frames,n=8):
    idx=np.linspace(0,len(frames)-1,min(n,len(frames))).round().astype(int)
    return [frames[i] for i in idx]

# ---- Stage1(실제 사용): MViT 입력용 클립 ----
def _crop_tensor(rgb,size=224):
    h,w=rgb.shape[:2]; scale=size/min(h,w)
    nh,nw=max(size,round(h*scale)),max(size,round(w*scale))
    rgb=cv2.resize(rgb,(nw,nh),interpolation=cv2.INTER_AREA)
    y,x=(nh-size)//2,(nw-size)//2
    return torch.from_numpy(rgb[y:y+size,x:x+size].copy()).permute(2,0,1).float()/255

def _clip(path,n=16,center=None):
    frames=load_frames(path); total=len(frames)
    if center is None: idx=np.linspace(0,total-1,n).round().astype(int)
    else: idx=np.clip(center-n//2+np.arange(n),0,total-1)
    x=torch.stack([_crop_tensor(frames[int(i)]) for i in idx],1)
    return x,total

# ---- Stage1(실험용, 미사용 - fit_stage1_experimental_handcrafted 전용): 재녹화 시뮬레이션 ----
def _moire_overlay(frame,rng):
    h,w=frame.shape[:2]; freq=rng.uniform(0.15,0.4); phase=rng.uniform(0,np.pi)
    yy,xx=np.mgrid[0:h,0:w]; grid=0.5+0.5*np.sin(freq*(xx+yy)+phase)
    strength=rng.uniform(6,18)
    out=frame.astype(np.float32)+(grid[...,None]-0.5)*strength
    return np.clip(out,0,255).astype(np.uint8)

def _flicker_stack(frames,rng):
    period=rng.uniform(3.5,9.0); amp=rng.uniform(0.06,0.16)
    return [np.clip(f.astype(np.float32)*(1+amp*np.sin(2*np.pi*i/period)),0,255).astype(np.uint8)
            for i,f in enumerate(frames)]

def _double_compress(frame,rng):
    quality=rng.randint(25,55); bgr=cv2.cvtColor(frame,cv2.COLOR_RGB2BGR)
    ok,buf=cv2.imencode('.jpg',bgr,[cv2.IMWRITE_JPEG_QUALITY,quality])
    bgr=cv2.imdecode(buf,cv2.IMREAD_COLOR)
    return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)

def _add_border(frame,rng):
    h,w=frame.shape[:2]; b=int(min(h,w)*rng.uniform(0.02,0.06)); out=frame.copy()
    out[:b]=out[-b:]=out[:,:b]=out[:,-b:]=0
    return out

def simulate_rerecording(frames,seed):
    rng=random.Random(seed)
    frames=_flicker_stack(frames,rng)
    frames=[_moire_overlay(f,rng) for f in frames]
    frames=[_double_compress(f,rng) for f in frames]
    frames=[_add_border(f,rng) for f in frames]
    return frames

def benign_augment(frames,seed):
    rng=random.Random(seed); gain=rng.uniform(0.85,1.15); quality=rng.randint(75,95); out=[]
    for f in frames:
        bright=np.clip(f.astype(np.float32)*gain,0,255).astype(np.uint8)
        bgr=cv2.cvtColor(bright,cv2.COLOR_RGB2BGR)
        ok,buf=cv2.imencode('.jpg',bgr,[cv2.IMWRITE_JPEG_QUALITY,quality])
        bgr=cv2.imdecode(buf,cv2.IMREAD_COLOR)
        out.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    return out

def _fft_high_freq_ratio(gray):
    f=np.fft.fftshift(np.fft.fft2(gray.astype(np.float32))); mag=np.abs(f)
    h,w=gray.shape; cy,cx=h//2,w//2
    yy,xx=np.mgrid[0:h,0:w]; r=np.sqrt((yy-cy)**2+(xx-cx)**2)
    radius=min(h,w)*0.15
    return float(mag[r>radius].sum()/(mag.sum()+1e-6))

def _blockiness(gray):
    g=gray.astype(np.float32); h,w=g.shape; h,w=h-h%8,w-w%8; g=g[:h,:w]
    boundary=np.abs(np.diff(g[:,7:w:8],axis=1)).mean() if w>8 else 0.0
    boundary+=np.abs(np.diff(g[7:h:8,:],axis=0)).mean() if h>8 else 0.0
    interior=np.abs(np.diff(g,axis=1)).mean()+np.abs(np.diff(g,axis=0)).mean()
    return float(boundary/(interior+1e-6))

def extract_features(frames):
    frames=sample_frames(frames,8)
    grays=[cv2.cvtColor(f,cv2.COLOR_RGB2GRAY) for f in frames]
    resized=[cv2.resize(g,(256,256)) for g in grays]
    fft_ratio=float(np.mean([_fft_high_freq_ratio(g) for g in resized]))
    brightness=np.array([g.mean() for g in grays],dtype=np.float32)
    flicker_std=float(brightness.std()/(brightness.mean()+1e-6))
    border=float(np.mean([np.concatenate([g[:4].ravel(),g[-4:].ravel(),g[:,:4].ravel(),g[:,-4:].ravel()]).mean() for g in grays]))
    blur=float(np.mean([cv2.Laplacian(g,cv2.CV_64F).var() for g in resized]))
    block=float(np.mean([_blockiness(g) for g in resized]))
    return np.array([fft_ratio,flicker_std,border,blur,block],dtype=np.float32)


In [ ]:
# ---- Stage1(실제 사용): MViTv2-S (팀 베이스라인 원안, 0.53397로 검증됨) ----
class Stage1MViT(nn.Module):
    def __init__(self):
        super().__init__(); self.net=mvit_v2_s(weights=None)
        self.net.head[1]=nn.Linear(self.net.head[1].in_features,2)
    def forward(self,x): return self.net(x)

# ---- Stage2: COCO 탐지기 + 재정렬기 ----
def load_detector():
    weights=FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT
    model=fasterrcnn_mobilenet_v3_large_320_fpn(weights=weights); model.eval()
    return model,weights.transforms(),weights.meta['categories']

@torch.inference_mode()
def detect_all_vehicles(model,transform,categories,frame,score_thr=0.2):
    x=transform(torch.from_numpy(frame).permute(2,0,1))
    out=model([x])[0]; candidates=[]
    for box,label,score in zip(out['boxes'],out['labels'],out['scores']):
        if score<score_thr or categories[label] not in VEHICLE_CLASSES: continue
        x0,y0,x1,y1=box.tolist(); candidates.append((x0,y0,x1,y1,float(score)))
    return candidates

def _rerank_features(c,w,h):
    x0,y0,x1,y1,score=c[:5]
    cx,cy=(x0+x1)/2/w,(y0+y1)/2/h; bw,bh=(x1-x0)/w,(y1-y0)/h
    return [cx,cy,bw,bh,score,bw/(bh+1e-6)]

def detect_vehicles(model,transform,categories,frame,reranker=None):
    candidates=detect_all_vehicles(model,transform,categories,frame,score_thr=SCORE_THR)
    if not candidates: return None
    if reranker is not None:
        net,mean,std=reranker; h,w=frame.shape[:2]
        feats=torch.tensor([_rerank_features(c,w,h) for c in candidates],dtype=torch.float32)
        with torch.inference_mode():
            scores=net((feats-mean)/std).squeeze(-1)
        return candidates[int(scores.argmax())]
    return max(candidates,key=lambda c:c[4]*(c[2]-c[0])*(c[3]-c[1]))

def motion_energy(frames):
    grays=[cv2.resize(cv2.cvtColor(f,cv2.COLOR_RGB2GRAY),(320,180)) for f in frames]
    diffs=[cv2.absdiff(grays[i],grays[i-1]).mean() for i in range(1,len(grays))]
    return np.array([diffs[0]]+diffs,dtype=np.float32)

def find_collision_frame(frames):
    # 시작부만 고정 8프레임 제외, 끝은 제외 안 함: CCD(801개 자차관여 실측)로 재보정.
    # 원래(비율 기반 margin=10%, 시작+끝 둘 다 제외)는 사고 클립 특성상 문제 있었음 -
    # 충돌이 늘 뒷부분(50프레임 중 30~49, 최솟값이 30!)이라 끝 10% 제외가 늦게 발생한
    # 진짜 충돌을 통째로 못 찾게 막았음. 비율 기반 margin=0(끝 제외 없음)도 시도했지만
    # DACON 공개 5샘플 중 하나(자차 무관 배경사고 영상)에서 초반 카메라 흔들림을
    # 충돌로 오검출(MAE 2.40->6.40). 영상 길이에 비례하는 "비율"보다 "고정 프레임 수"가
    # 더 타당하다고 보고 시작 8프레임만 고정 제외 + 끝 제외 없음으로 재시도: CCD 기준
    # MAE 6.22(최선), within±3 50.9%, 공개 5샘플에서도 오검출 없음(MAE 2.40 유지).
    energy=motion_energy(frames)
    start_exclude=min(8,max(0,len(energy)-1))
    window=energy[start_exclude:]
    return int(np.argmax(window))+start_exclude

# ---- Stage2 진입시점: 차선 추정(entry_frame 공식 정의 - "피해차량 바퀴가 피의차량
# 차선에 최초로 닿는 시점", talkboard 417186/417277) ----
def _fit_lane_side(points):
    if len(points)<2: return None
    ys=np.array([p[1] for p in points],dtype=np.float64); xs=np.array([p[0] for p in points],dtype=np.float64)
    if ys.std()<1e-3: return None
    m,b=np.polyfit(ys,xs,1); return float(m),float(b)

# Ultra-Fast-Lane-Detection(ECCV 2020, cfzd/Ultra-Fast-Lane-Detection, MIT, TuSimple res18).
# 2026-09-13: 기존 Canny+HoughLinesP는 DACON 공개 5샘플 전부에서 실패(talkboard 417288).
# 사전학습 딥러닝으로 교체 - 공개 5샘플 10프레임 샘플링 기준 검출률 7,2,1,2,5/10
# (Hough는 0/10). model/stage2/ufld_tusimple.pth 없으면 Hough로 폴백.
UFLD_ROW_ANCHOR=[64,68,72,76,80,84,88,92,96,100,104,108,112,116,120,124,128,132,136,140,144,148,
                 152,156,160,164,168,172,176,180,184,188,192,196,200,204,208,212,216,220,224,228,
                 232,236,240,244,248,252,256,260,264,268,272,276,280,284]
UFLD_GRIDING_NUM=100; UFLD_CLS_NUM_PER_LANE=56; UFLD_NUM_LANES=4
_UFLD_TRANSFORM=tv_transforms.Compose([
    tv_transforms.Resize((288,800)), tv_transforms.ToTensor(),
    tv_transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

class _UFLDNet(nn.Module):
    """cfzd/Ultra-Fast-Lane-Detection의 parsingNet(backbone=res18)과 state_dict 호환되는
    최소 재구현 - 원본 레포 전체를 의존성으로 들이지 않기 위해 추론에 필요한 부분만."""
    def __init__(self):
        super().__init__()
        self.model=resnet18(weights=None); self.pool=nn.Conv2d(512,8,1)
        self.cls=nn.Sequential(nn.Linear(1800,2048),nn.ReLU(),nn.Linear(2048,22624))
    def forward(self,x):
        m=self.model
        x=m.conv1(x); x=m.bn1(x); x=m.relu(x); x=m.maxpool(x)
        x=m.layer1(x); x=m.layer2(x); x=m.layer3(x); x=m.layer4(x)
        x=self.pool(x); x=torch.flatten(x,1); x=self.cls(x)
        return x.view(-1,UFLD_GRIDING_NUM+1,UFLD_CLS_NUM_PER_LANE,UFLD_NUM_LANES)

def load_ufld(path=None):
    path=Path(path) if path else MODEL/'stage2'/'ufld_tusimple.pth'
    if not path.exists(): return None
    net=_UFLDNet()
    state=torch.load(path,map_location='cpu',weights_only=False)
    net.load_state_dict(state['model'] if 'model' in state else state, strict=False)
    net.eval(); return net

@torch.inference_mode()
def _detect_lane_boundaries_ufld(frame,net):
    h,w=frame.shape[:2]
    x=_UFLD_TRANSFORM(Image.fromarray(frame))
    out=net(x[None])[0].numpy()
    out=out[:,::-1,:]
    exp=np.exp(out[:-1]-out[:-1].max(axis=0,keepdims=True))
    prob=exp/exp.sum(axis=0,keepdims=True)
    idx=(np.arange(UFLD_GRIDING_NUM)+1).reshape(-1,1,1)
    loc=(prob*idx).sum(axis=0)
    loc[out.argmax(axis=0)==UFLD_GRIDING_NUM]=0
    col_sample_w=799.0/(UFLD_GRIDING_NUM-1)
    lanes=[]
    for lane_i in range(UFLD_NUM_LANES):
        if np.sum(loc[:,lane_i]!=0)<=2: continue
        pts=[]
        for k in range(UFLD_CLS_NUM_PER_LANE):
            if loc[k,lane_i]>0:
                x_px=loc[k,lane_i]*col_sample_w*w/800-1
                y_px=h*(UFLD_ROW_ANCHOR[UFLD_CLS_NUM_PER_LANE-1-k]/288)-1
                pts.append((x_px,y_px))
        if len(pts)>=2: lanes.append(pts)
    if len(lanes)<2: return None
    centers=[float(np.mean([p[0] for p in pts])) for pts in lanes]
    order=np.argsort(centers)
    lanes_sorted=[lanes[i] for i in order]; centers_sorted=[centers[i] for i in order]
    straddle=None
    for i in range(len(centers_sorted)-1):
        if centers_sorted[i]<=w/2<=centers_sorted[i+1]: straddle=i; break
    if straddle is None:
        a,b=sorted(np.argsort([abs(c-w/2) for c in centers_sorted])[:2].tolist())
        left_pts,right_pts=lanes_sorted[a],lanes_sorted[b]
    else:
        left_pts,right_pts=lanes_sorted[straddle],lanes_sorted[straddle+1]
    left,right=_fit_lane_side(left_pts),_fit_lane_side(right_pts)
    if left is None or right is None: return None
    return left,right

def _detect_lane_boundaries_hough(frame):
    # 사다리꼴 ROI(화면 하단 30%, 원근상 이보다 멀면 차선이 거의 수평이 돼 구분 불가)
    # 로 도로 밖 직선을 배제하고 Canny+HoughLinesP, 좌/우 기울기 부호로 분리해 피팅.
    # ufld_tusimple.pth가 없을 때만의 폴백 - DACON 공개 5샘플 전부에서 실패했던 방식.
    h,w=frame.shape[:2]; roi_top=int(h*0.7)
    mask=np.zeros((h,w),dtype=np.uint8)
    trapezoid=np.array([[0,h],[int(w*0.3),roi_top],[int(w*0.7),roi_top],[w,h]])
    cv2.fillPoly(mask,[trapezoid],255)
    gray=cv2.cvtColor(frame,cv2.COLOR_RGB2GRAY)
    edges=cv2.bitwise_and(cv2.Canny(gray,50,150),mask)
    lines=cv2.HoughLinesP(edges,1,np.pi/180,threshold=25,minLineLength=int(h*0.08),maxLineGap=30)
    if lines is None: return None
    left_pts,right_pts=[],[]
    for x1,y1,x2,y2 in lines.reshape(-1,4):
        if x2==x1: continue
        slope=(y2-y1)/(x2-x1)
        if abs(slope)<0.4: continue
        (left_pts if slope<0 else right_pts).extend([(x1,y1),(x2,y2)])
    left,right=_fit_lane_side(left_pts),_fit_lane_side(right_pts)
    if left is None or right is None: return None
    return left,right

def detect_lane_boundaries(frame,ufld_net=None):
    if ufld_net is not None: return _detect_lane_boundaries_ufld(frame,ufld_net)
    return _detect_lane_boundaries_hough(frame)

def _default_lane(h,w):
    # 검출 실패시 기본값(DACON 공개 5샘플 전부 실패 - talkboard 417288 참고) - 자차가
    # 차선 중앙, 차선폭은 화면폭의 38%로 가정.
    half=0.19*w
    left=tuple(np.polyfit([h,0],[w/2-half,w/2],1)); right=tuple(np.polyfit([h,0],[w/2+half,w/2],1))
    return left,right

def estimate_ego_lane(frames,ts,h,w,ufld_net=None):
    ts=list(ts); lefts,rights=[],[]
    for t in ts:
        fit=detect_lane_boundaries(frames[t],ufld_net=ufld_net)
        if fit is None: continue
        lefts.append(fit[0]); rights.append(fit[1])
    if len(lefts)<max(2,len(ts)//4): return _default_lane(h,w)
    return tuple(np.median(lefts,axis=0)),tuple(np.median(rights,axis=0))

def _lane_x_at(line,y):
    m,b=line; return m*y+b

def _crosses_into_lane(box,lane):
    x0,_,x1,y1=box[:4]
    lx,rx=_lane_x_at(lane[0],y1),_lane_x_at(lane[1],y1)
    if lx>rx: lx,rx=rx,lx
    return x1>lx and x0<rx

def _evasion_space_from_lane(frame,lane,model,transform,categories,y_ref,w):
    # 공식 정의(talkboard 417319 - "자차가 진행방향을 바꿔 피할 수 있는 물리적 공간")에
    # 맞춰 상대차량 박스 좌우 여백(자차와 무관한 기준이었음) 대신 자차 차선 경계 바로
    # 옆 인접공간의 차량 점유 여부로 판단. 한계: 반대차선/보도 구분할 도로 정보가 없어
    # 화면 가장자리 배제만 하는 거친 근사(정량 검증 불가, 눈검증만).
    lx,rx=_lane_x_at(lane[0],y_ref),_lane_x_at(lane[1],y_ref)
    if lx>rx: lx,rx=rx,lx
    lane_w=max(rx-lx,1.0)
    others=detect_all_vehicles(model,transform,categories,frame,score_thr=SCORE_THR)
    def occupied(x0,x1): return any(not (c[2]<x0 or c[0]>x1) for c in others)
    left_clear=lx>0.05*w and not occupied(max(0.0,lx-lane_w),lx)
    right_clear=rx<0.95*w and not occupied(rx,min(w,rx+lane_w))
    return int(left_clear or right_clear)

def find_entry_and_scene(model,transform,categories,frames,collision_frame,reranker=None,ufld_net=None):
    h,w=frames[0].shape[:2]
    window_lo=max(0,collision_frame-90); window_hi=min(len(frames)-1,collision_frame+5)
    detections={}
    for t in range(window_lo,window_hi+1):
        det=detect_vehicles(model,transform,categories,frames[t],reranker=reranker)
        if det is not None: detections[t]=det
    lane=estimate_ego_lane(frames,detections.keys(),h,w,ufld_net=ufld_net) if detections else _default_lane(h,w)
    entry_frame,entry_side=None,None
    for t in sorted(t for t in detections if t<=collision_frame):
        det=detections[t]; x0,_,x1,_,_=det
        if _crosses_into_lane(det,lane):
            entry_frame=t; entry_side='LEFT' if (x0+x1)/2<w/2 else 'RIGHT'; break
    if entry_frame is None: entry_frame,entry_side=window_lo,'RIGHT'  # window_lo=0이면 "시작 전 진입" 규칙과 일치
    box_at_collision=None
    if detections:
        nearest_t=min(detections,key=lambda t:abs(t-collision_frame)); box_at_collision=detections[nearest_t]
    evasion_space=0
    if box_at_collision is not None:
        _,y0,_,y1,_=box_at_collision
        evasion_space=_evasion_space_from_lane(frames[nearest_t],lane,model,transform,categories,y_ref=y1,w=w)
    return entry_frame,entry_side,evasion_space

# ---- Stage3: optical flow (학습 없음, 임계값만 보정) ----
def compute_flow_series(frames):
    small=[cv2.resize(cv2.cvtColor(f,cv2.COLOR_RGB2GRAY),FLOW_SIZE) for f in frames]
    n=len(small)//2; w,h=FLOW_SIZE
    road=slice(int(h*0.55),h); horizon=slice(int(h*0.25),int(h*0.55))
    speed=np.zeros(n,dtype=np.float32); steer=np.zeros(n,dtype=np.float32)
    for t in range(n):
        i0=min(2*t,len(small)-3); i1=i0+2
        flow=cv2.calcOpticalFlowFarneback(small[i0],small[i1],None,0.5,2,15,3,5,1.2,0)
        mag=np.sqrt(flow[...,0]**2+flow[...,1]**2)
        speed[t]=float(np.median(mag[road])); steer[t]=float(np.median(flow[horizon,:,0]))
    return speed,steer

def _smooth(x,k=3):
    if len(x)<2*k+1: return x
    width=2*k+1; padded=np.pad(x,(k,k),mode='edge')
    return np.median(np.lib.stride_tricks.sliding_window_view(padded,width),axis=-1)

def classify(speed,steer,stopped_thr,accel_eps,steer_thr):
    # steer 스무딩 추가했다가(LOVO 0.670->0.710) 실제 점수가 0.521->0.48->0.47로
    # 계속 떨어져서 원복. Macro-F1 공식(팀 이슈 #3: 0.7*accel+0.3*steer, STOPPED 제외)
    # 으로 다시 그리드서치해도 스무딩 여부와 무관하게 같은 임계값이 최적이라, 스무딩
    # 자체의 문제라기보다 "공개 5비디오 로컬검증이 실제 숨은 평가셋과 거의 무관하다"는
    # 구조적 한계로 보임(팀장님도 반대방향 동일 현상 - 이슈 #10). 실측 검증된 상태로 복귀.
    # 2026-09-13: 스무딩을 평균->중앙값으로 교체(팀원 정민 submit-3 참고, 공개 50라벨 acc
    # 변화 없이 회귀 없음 확인 - 이 함수는 20fps 공개영상용이라 실제 gap=1 fix는 추론
    # 노트북에만 적용, 본 함수는 참고/재현용).
    speed_s=_smooth(speed); n=len(speed_s); accel_out,steer_out=[],[]
    for t in range(n):
        if speed_s[t]<stopped_thr: accel_out.append('STOPPED')
        else:
            lo,hi=max(0,t-3),min(n,t+4)
            width=(hi-1)-lo
            slope=(speed_s[hi-1]-speed_s[lo])*(6/width) if width>0 else 0.0
            accel_out.append('ACCELERATING' if slope>accel_eps else 'DECELERATING' if slope<-accel_eps else 'CONSTANT')
        s=steer[t]
        # 좌회전->배경이 화면에서 오른쪽으로 흐름(flow_x 양수). AIHub 실측 자이로+실제 프레임으로 검증된 부호.
        steer_out.append('LEFT' if s>steer_thr else 'RIGHT' if s<-steer_thr else 'STRAIGHT')
    return accel_out,steer_out

In [ ]:
def _video_bpp(path):
    """size(bytes)*8 / (frame_count*width*height) - 해상도/길이 무관 압축난이도 지표.
    공개 10샘플에서 원본 vs 재녹화가 깨끗하게 갈림(LOO 9~10/10) - 단, DACON이
    재녹화 예제는 "실제 재촬영 아닌 파생 예제"라 명시해서 실제 평가셋에 안 통할
    위험 있음(인코더/코덱 차이가 DACON 예제 생성 파이프라인 특성일 수 있음).
    그래도 MViT(LOO 0/10, 페어 암기로 역방향)보다 훨씬 강한 로컬 신호라 공격적으로
    채택 - MViT를 보조(가중치 0.25)로만 섞어서 한쪽에만 전부 걸지는 않음."""
    cap=cv2.VideoCapture(str(path))
    nframes=cap.get(cv2.CAP_PROP_FRAME_COUNT); w=cap.get(cv2.CAP_PROP_FRAME_WIDTH); h=cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    if nframes<=0 or w<=0 or h<=0: return 0.0
    return Path(path).stat().st_size*8/(nframes*w*h)

def fit_stage1():
    """팀 베이스라인 원안(MViTv2-S, ImageNet 가중치 없이 5+5 공개샘플로 1 epoch 학습)
    + bpp(비트레이트) 신호, 가중치 0.75(원복).
    LOO 스윕(stage1_blend_sweep.py)은 weight=1.0(순수 bpp)이 9/10으로 0.75(8/10)보다
    낫다고 나와서 한 번 1.0으로 올렸으나, 실제 제출 결과 0.589(0.75) -> 0.5783(1.0)로
    오히려 하락 - 로컬 LOO(10샘플)가 이번에도 실제 방향과 어긋남(공개 소량 검증의
    반복되는 한계). 실측 A/B(0.75 real 0.589 > 1.0 real 0.5783)를 우선해서 0.75로
    되돌림. MViT 학습/추론 코드는 그대로 둠."""
    out=MODEL/'stage1'; out.mkdir(parents=True,exist_ok=True)
    df=pd.read_csv(DATA/'stage1/labels.csv')
    model=Stage1MViT().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),1e-4)
    model.train()
    for r in df.sample(frac=1,random_state=20260825).itertuples():
        x,_=_clip(DATA/'stage1'/r.path,16); x=(x-S1_MEAN)/S1_STD
        y=torch.tensor([0 if r.label=='ORIGINAL' else 1],device=DEVICE)
        loss=nn.functional.cross_entropy(model(x[None].to(DEVICE)),y)
        opt.zero_grad(); loss.backward(); opt.step()

    bpps={r.ID:_video_bpp(DATA/'stage1'/r.path) for r in df.itertuples()}
    o_vals=[bpps[r.ID] for r in df.itertuples() if r.label=='ORIGINAL']
    r_vals=[bpps[r.ID] for r in df.itertuples() if r.label=='RERECORDED']
    bpp_thr=(max(o_vals)+min(r_vals))/2; bpp_scale=max((min(r_vals)-max(o_vals))/2,1e-6)
    print(f'  stage1 bpp_thr={bpp_thr:.5f} bpp_scale={bpp_scale:.5f} (원본 bpp<={max(o_vals):.4f}, 재녹화 bpp>={min(r_vals):.4f})')

    torch.save({'model':model.net.state_dict(),'size':224,'frames':16,
                'bpp_thr':bpp_thr,'bpp_scale':bpp_scale,'bpp_weight':0.75}, out/'best.pt')

def fit_stage1_experimental_handcrafted():
    """실험용(기본 실행 대상 아님) - 합성 재녹화+benign 증강으로 만든 데이터로 5개 핸드크래프트
    피처+로지스틱회귀 학습. 로컬 합성데이터 정확도는 95~98%였지만 실제 제출 0.316으로
    베이스라인(0.53397)보다 나빴다 - 합성 artefact 과적합으로 추정. 실제 재녹화 데이터
    확보 전까지는 model/stage1/best.pt를 덮어쓰지 않도록 별도 경로에 저장한다."""
    out=MODEL/'stage1_experimental'; out.mkdir(parents=True,exist_ok=True)
    originals=sorted((DATA/'stage1/original').glob('*.mp4'))+sorted((DATA/'stage2/videos').glob('*.mp4'))+sorted((DATA/'stage3/videos').glob('*.mp4'))
    rerecorded=sorted((DATA/'stage1/rerecorded').glob('*.mp4'))
    X,y=[],[]
    for path in originals:
        frames=sample_frames(load_frames(path),8)
        X.append(extract_features(frames)); y.append(0)
        for v in range(3):
            X.append(extract_features(benign_augment(frames,seed=hash((path.name,v))&0xFFFF))); y.append(0)
            X.append(extract_features(simulate_rerecording(frames,seed=hash((path.name,v,'r'))&0xFFFF))); y.append(1)
    for path in rerecorded:
        X.append(extract_features(load_frames(path))); y.append(1)
    X,y=np.stack(X),np.array(y,dtype=np.float32)

    mean,std=X.mean(0),X.std(0)+1e-6
    Xn=torch.tensor((X-mean)/std,dtype=torch.float32); yt=torch.tensor(y)
    weight=torch.zeros(X.shape[1],requires_grad=True); bias=torch.zeros(1,requires_grad=True)
    opt=torch.optim.Adam([weight,bias],lr=0.1)
    for _ in range(300):
        loss=nn.functional.binary_cross_entropy_with_logits(Xn@weight+bias,yt)
        opt.zero_grad(); loss.backward(); opt.step()
    acc=float(((Xn@weight+bias>0).float()==yt).float().mean())
    print(f'  stage1(실험용) train acc: {acc:.3f} (합성 데이터 기준 - 실제 0.316으로 검증됨, 참고용)')
    torch.save({'weight':weight.detach(),'bias':bias.detach(),'feat_mean':mean,'feat_std':std}, out/'best.pt')

def fit_stage2():
    """COCO 사전학습 탐지기는 그대로 저장(파인튜닝 아님). 재정렬기(reranker.pt)는 AIHub
    실라벨(597, 차대차 카테고리)로 별도 학습한 산출물 - 공개 5샘플엔 그런 라벨이 없어서
    여기서 재현 불가. 이미 model/stage2/reranker.pt가 있으면(로컬에서 미리 학습) 그대로 두고,
    없으면 score*area 폴백으로 동작 (detect_vehicles 참고).

    ufld_tusimple.pth(차선검출, cfzd/Ultra-Fast-Lane-Detection MIT license)는 external/ufld/
    tusimple_18.pth에 미리 받아둔 걸 model/stage2/로 복사만 한다(학습 아님, 사전학습
    그대로 사용) - 없으면 Hough 폴백으로 동작(detect_lane_boundaries 참고)."""
    out=MODEL/'stage2'; out.mkdir(parents=True,exist_ok=True)
    model,_,_=load_detector()
    torch.save(model.state_dict(), out/'detector.pth')
    reranker_path=out/'reranker.pt'
    print(f"  reranker.pt {'존재 - 유지' if reranker_path.exists() else '없음 - score*area 폴백으로 동작'}")

    ufld_dst=out/'ufld_tusimple.pth'; ufld_src=ROOT/'external'/'ufld'/'tusimple_18.pth'
    if not ufld_dst.exists() and ufld_src.exists(): shutil.copy2(ufld_src,ufld_dst)
    print(f"  ufld_tusimple.pth {'존재 - 차선검출 사용' if ufld_dst.exists() else '없음 - Hough 폴백으로 동작'}")

def fit_stage3():
    """stopped_thr/accel_eps: AIHub 실측 CAN(10영상/35982프레임, LOVO 0.687) 기준 -
    실제 제출로 0.521->0.53113 개선 확인됨(real validated).
    steer_thr: SullyChen driving_dataset(45406프레임, 실측 조향각 degree) 기준으로 교체
    (2026-09-12) - AIHub CAN의 steering_angle 필드가 전 영상·전 차종에서 100% 0으로
    죽어있어서(placeholder) 각속도로 대체했었는데, 진짜 조향각 데이터로 재보정한 것.
    optical-flow steer_proxy와의 상관계수 -0.845(AIHub 각속도 -0.519보다 강함). steer_thr
    그리드서치: 이전 0.464는 정확도 0.865, 0.65가 0.886으로 더 나음 - 아직 실제 제출로는
    미검증(다음 제출 대상). 이전 값들은 stage3_can_candidate.PREV_CAN_CANDIDATE 참고."""
    out=MODEL/'stage3'; out.mkdir(parents=True,exist_ok=True)
    params={'stopped_thr':0.1,'accel_eps':0.01,'steer_thr':0.65}  # steer_thr만 SullyChen으로 재보정
    print(f'  stage3 보정값 적용: {params} (steer_thr는 SullyChen 45406프레임 검증, acc 0.886)')
    torch.save(params, out/'best.pt')

def fit_stage3_grid_search_50samples():
    """원래 로직(참고/원복용, 기본 실행 대상 아님) - 공개 라벨 50개(6초 간격) 기준 그리드서치.
    steer 스무딩 추가가 로컬(LOVO)엔 나았지만(0.670->0.710) 실제 제출은 0.521->0.48로
    떨어져서 되돌린 이력 있음(그리드도 6x6x6가 10x10x10보다 실측 안전). 결과값
    (stopped_thr=0.1, accel_eps=0.3, steer_thr=0.28)이 real score 0.521로 검증된 상태
    (CAN 기반 fit_stage3()의 0.53113보다 낮음 - 이제는 CAN 기반이 기본)."""
    out=MODEL/'stage3'; out.mkdir(parents=True,exist_ok=True)
    labels=pd.read_csv(DATA/'stage3/labels.csv')
    cache={}
    for vid_id,group in labels.groupby('ID'):
        frames=load_frames(DATA/'stage3/videos'/f'{vid_id}.mp4')
        cache[vid_id]=(compute_flow_series(frames),group)

    best=None
    grid=itertools.product(np.linspace(0.1,1.5,6), np.linspace(0.02,0.3,6), np.linspace(0.1,1.0,6))
    for stopped_thr,accel_eps,steer_thr in grid:
        correct=total=0
        for vid_id,((speed,steer),group) in cache.items():
            accel_pred,steer_pred=classify(speed,steer,stopped_thr,accel_eps,steer_thr)
            for row in group.itertuples():
                idx=min(row.sample_index,len(accel_pred)-1); total+=2
                correct+=accel_pred[idx]==row.accel_label; correct+=steer_pred[idx]==row.steer_label
        acc=correct/total
        if best is None or acc>best[0]: best=(acc,stopped_thr,accel_eps,steer_thr)
    acc,stopped_thr,accel_eps,steer_thr=best
    print(f'  stage3 calibrated acc: {acc:.3f} (stopped_thr={stopped_thr:.3f}, accel_eps={accel_eps:.3f}, steer_thr={steer_thr:.3f})')
    torch.save({'stopped_thr':stopped_thr,'accel_eps':accel_eps,'steer_thr':steer_thr}, out/'best.pt')

## 2. Stage 1·2·3 학습

In [ ]:
print('device:',DEVICE)
fit_stage1(); print('Stage 1 완료')
fit_stage2(); print('Stage 2 완료')
fit_stage3(); print('Stage 3 완료')

In [ ]:
for p in sorted((ROOT/'model').rglob('*')):
    if p.is_file(): print(p.relative_to(ROOT),f'{p.stat().st_size/1024**2:.1f} MB')